# Auction Condition Ladder Pricing

Sparse realized sales are normal in auctions.

You may have one sale in **Near Mint**, two in **Good**, nothing in **Excellent**, and one deeply suspicious listing labeled **"storage unit fresh"**.

This notebook helps estimate prices across a user-defined condition ladder when realized sales only exist for some grades.

It is designed for auction-style markets where:

- Products sell irregularly
- Conditions are inconsistent or user-defined
- Some condition grades have no realized sales
- You still need a reasonable price ladder before making a pricing, reserve, bid, or listing decision

The notebook is intentionally practical. It does not pretend that four auction sales and a condition label can reveal the hidden structure of the universe.

## How this notebook works

You define a condition ladder.

Example:

| condition | rank |
|---|---:|
| Poor | 1 |
| Fair | 2 |
| Good | 3 |
| Very Good | 4 |
| Excellent | 5 |
| Near Mint | 6 |

The notebook then:

1. Loads realized auction sales or creates simulated data
2. Maps condition labels to your ladder
3. Estimates a condition-price relationship
4. Fills missing condition prices for each product
5. Flags where estimates are based on thin evidence
6. Exports the result to Excel

The goal is not certainty. The goal is a defensible starting point that is better than shrugging at the screen with confidence.

In [1]:
# ============================================================
# 1. CONFIG
# Edit this section first.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import math

# Optional input file.
# Supported: csv, xlsx, xls, json, parquet.
# Leave as None to run the built-in simulated auction dataset.
INPUT_FILE = None
# INPUT_FILE = "auction_sales.csv"

OUTPUT_EXCEL = "auction_condition_ladder_pricing_output.xlsx"

# Define your own condition ladder here.
# Higher rank should mean better condition.
CONDITION_LADDER = [
    {"condition": "Poor", "rank": 1},
    {"condition": "Fair", "rank": 2},
    {"condition": "Good", "rank": 3},
    {"condition": "Very Good", "rank": 4},
    {"condition": "Excellent", "rank": 5},
    {"condition": "Near Mint", "rank": 6},
]

# Column names expected after loading.
# The loader tries to infer these, but you can override them here.
COLUMN_MAP = {
    "product": None,      # Example: "product", "sku", "model", "item_type"
    "condition": None,    # Example: "condition", "grade", "quality"
    "price": None,        # Example: "realized_price", "sale_price", "hammer_price"
    "date": None,         # Optional: "auction_date", "sale_date"
}

# Pricing controls.
MIN_OBS_FOR_PRODUCT_CURVE = 2
GLOBAL_FALLBACK_SLOPE = 0.08
# 0.08 means roughly 8.3 percent higher price per one condition-rank step in log space.

ROUND_TO = 1

## Data requirements

Minimum useful columns:

- Product identifier, such as SKU, model, item type, title cluster, or catalog number
- Condition label
- Realized sale price

Optional:

- Sale date
- Auction venue
- Lot count
- Seller
- Notes

The more consistent the product grouping, the better the estimates. If one "product" value mixes different editions, sizes, years, or variants, the model will obediently price nonsense. Computers remain very literal employees.

In [2]:
# ============================================================
# 2. LOAD DATA OR CREATE SIMULATED AUCTION SALES
# ============================================================

def make_simulated_sales(seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)

    products = {
        "Vintage Camera A": 220,
        "Vintage Camera B": 375,
        "Watch Model X": 900,
        "Watch Model Y": 1250,
        "Collectible Card Z": 140,
        "Signed Print Q": 520,
    }

    ladder = pd.DataFrame(CONDITION_LADDER)
    rows = []

    # Sparse by design: some products sell in only a few conditions.
    for product, base in products.items():
        observed_ranks = rng.choice(ladder["rank"].values, size=rng.integers(2, 5), replace=False)
        for rank in sorted(observed_ranks):
            condition = ladder.loc[ladder["rank"] == rank, "condition"].iloc[0]
            condition_effect = math.exp(0.09 * (rank - ladder["rank"].median()))
            noise = rng.lognormal(mean=0, sigma=0.10)
            price = base * condition_effect * noise
            rows.append({
                "product": product,
                "condition": condition,
                "realized_price": round(price, 2),
                "auction_date": pd.Timestamp("2025-01-01") + pd.Timedelta(days=int(rng.integers(0, 365))),
            })

    return pd.DataFrame(rows)


def read_any_file(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"Unsupported file type: {suffix}")


def infer_column(df: pd.DataFrame, candidates: list) -> str | None:
    lower_map = {str(c).lower().strip(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for col in df.columns:
        col_l = str(col).lower()
        if any(cand.lower() in col_l for cand in candidates):
            return col

    return None


if INPUT_FILE is None:
    sales_raw = make_simulated_sales()
    print("No input file provided. Using simulated auction sales. The fake data is behaving better than some real datasets.")
else:
    sales_raw = read_any_file(INPUT_FILE)
    print(f"Loaded: {INPUT_FILE}")

sales_raw.head()

No input file provided. Using simulated auction sales. The fake data is behaving better than some real datasets.


,product,condition,realized_price,auction_date
0,Vintage Camera A,Very Good,248.06,2025-02-01
1,Vintage Camera A,Near Mint,226.68,2025-09-12
2,Vintage Camera B,Very Good,391.60,2025-11-03
3,Vintage Camera B,Excellent,468.65,2025-06-14
4,Vintage Camera B,Near Mint,507.61,2025-10-13


In [3]:
# ============================================================
# 3. MAP COLUMNS AND CLEAN INPUT
# ============================================================

def prepare_sales_data(df: pd.DataFrame, column_map: dict) -> pd.DataFrame:
    df = df.copy()

    product_col = column_map.get("product") or infer_column(df, [
        "product", "sku", "model", "item", "item_type", "title", "catalog", "lot_name"
    ])
    condition_col = column_map.get("condition") or infer_column(df, [
        "condition", "grade", "quality", "state"
    ])
    price_col = column_map.get("price") or infer_column(df, [
        "realized_price", "sale_price", "hammer_price", "price", "amount", "sold_for"
    ])
    date_col = column_map.get("date") or infer_column(df, [
        "auction_date", "sale_date", "date", "sold_date"
    ])

    missing = {
        "product": product_col,
        "condition": condition_col,
        "price": price_col,
    }
    missing = [k for k, v in missing.items() if v is None]
    if missing:
        raise ValueError(
            "Could not infer required columns: "
            + ", ".join(missing)
            + ". Set COLUMN_MAP manually. Yes, the notebook needs columns. It is needy in that way."
        )

    out = pd.DataFrame({
        "product": df[product_col].astype(str).str.strip(),
        "condition": df[condition_col].astype(str).str.strip(),
        "realized_price": pd.to_numeric(df[price_col], errors="coerce"),
    })

    if date_col is not None:
        out["auction_date"] = pd.to_datetime(df[date_col], errors="coerce")
    else:
        out["auction_date"] = pd.NaT

    out = out.dropna(subset=["product", "condition", "realized_price"])
    out = out[out["realized_price"] > 0].copy()

    return out


sales = prepare_sales_data(sales_raw, COLUMN_MAP)

ladder = pd.DataFrame(CONDITION_LADDER).copy()
ladder["condition"] = ladder["condition"].astype(str).str.strip()
ladder["rank"] = pd.to_numeric(ladder["rank"], errors="coerce")

if ladder["condition"].duplicated().any():
    raise ValueError("Condition ladder contains duplicate condition labels.")

if ladder["rank"].isna().any():
    raise ValueError("Condition ladder contains non-numeric ranks.")

sales = sales.merge(ladder, on="condition", how="left")

unmapped = sales[sales["rank"].isna()]["condition"].unique()
if len(unmapped):
    raise ValueError(
        "These condition labels were not found in CONDITION_LADDER: "
        + ", ".join(map(str, unmapped))
        + ". Add them to the ladder or clean the labels."
    )

sales["log_price"] = np.log(sales["realized_price"])

print(f"Clean realized sales: {len(sales):,}")
print(f"Products: {sales['product'].nunique():,}")
print(f"Condition grades in ladder: {len(ladder):,}")

sales.head()

Clean realized sales: 18
Products: 6
Condition grades in ladder: 6


,product,condition,realized_price,auction_date,rank,log_price
0,Vintage Camera A,Very Good,248.06,2025-02-01,4,5.513671
1,Vintage Camera A,Near Mint,226.68,2025-09-12,6,5.423539
2,Vintage Camera B,Very Good,391.60,2025-11-03,4,5.970241
3,Vintage Camera B,Excellent,468.65,2025-06-14,5,6.149856
4,Vintage Camera B,Near Mint,507.61,2025-10-13,6,6.229713


## Estimate the condition curve

This notebook uses log prices because condition premiums tend to behave multiplicatively.

A move from **Good** to **Very Good** is usually better described as a percentage change than a fixed dollar change.

The model estimates a global condition slope from within-product observed sales where possible. If the data is too thin, it falls back to the configured default.

This is not magic. It is structured interpolation with receipts.

In [4]:
# ============================================================
# 4. ESTIMATE GLOBAL CONDITION EFFECT
# ============================================================

def estimate_global_condition_slope(sales: pd.DataFrame, fallback: float = GLOBAL_FALLBACK_SLOPE) -> dict:
    slopes = []

    # Pairwise within-product slopes in log-price space.
    for product, g in sales.groupby("product"):
        med = (
            g.groupby("rank", as_index=False)
             .agg(median_log_price=("log_price", "median"), n=("log_price", "size"))
             .sort_values("rank")
        )

        if len(med) < 2:
            continue

        rows = med.to_dict("records")
        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):
                rank_diff = rows[j]["rank"] - rows[i]["rank"]
                if rank_diff == 0:
                    continue
                slope = (rows[j]["median_log_price"] - rows[i]["median_log_price"]) / rank_diff
                if np.isfinite(slope):
                    slopes.append(slope)

    if len(slopes) == 0:
        return {
            "global_slope": fallback,
            "slope_source": "fallback",
            "n_pairwise_slopes": 0,
        }

    slopes = np.array(slopes, dtype=float)
    # Robust central tendency. Auctions have outliers. This is shocking to absolutely no one.
    slope = float(np.median(slopes))

    return {
        "global_slope": slope,
        "slope_source": "observed_pairwise_median",
        "n_pairwise_slopes": int(len(slopes)),
    }


slope_info = estimate_global_condition_slope(sales)
slope_info

{'global_slope': 0.07933049301126831,
 'slope_source': 'observed_pairwise_median',
 'n_pairwise_slopes': 20}

In [5]:
# ============================================================
# 5. BUILD PRODUCT CONDITION PRICE LADDER
# ============================================================

def build_condition_price_ladder(
    sales: pd.DataFrame,
    ladder: pd.DataFrame,
    global_slope: float,
    min_obs_for_product_curve: int = MIN_OBS_FOR_PRODUCT_CURVE,
    round_to: int = ROUND_TO,
) -> pd.DataFrame:
    all_rows = []

    ladder_sorted = ladder.sort_values("rank").reset_index(drop=True)

    for product, g in sales.groupby("product"):
        anchor_rank = float(np.average(g["rank"], weights=np.ones(len(g))))
        anchor_log_price = float(np.median(g["log_price"]))

        product_slope = global_slope
        slope_source = "global"

        med_by_rank = (
            g.groupby("rank", as_index=False)
             .agg(median_log_price=("log_price", "median"))
             .sort_values("rank")
        )

        if len(med_by_rank) >= min_obs_for_product_curve:
            x = med_by_rank["rank"].to_numpy(dtype=float)
            y = med_by_rank["median_log_price"].to_numpy(dtype=float)
            if len(np.unique(x)) >= 2:
                product_slope = float(np.polyfit(x, y, deg=1)[0])
                slope_source = "product_specific"

        for _, r in ladder_sorted.iterrows():
            target_rank = float(r["rank"])
            est_log = anchor_log_price + product_slope * (target_rank - anchor_rank)
            est_price = math.exp(est_log)

            all_rows.append({
                "product": product,
                "condition": r["condition"],
                "rank": target_rank,
                "estimated_price": round(est_price, round_to),
                "slope_used": product_slope,
                "slope_source": slope_source,
                "anchor_rank": anchor_rank,
                "product_observed_sales": len(g),
            })

    result = pd.DataFrame(all_rows)

    result = result.merge(
        sales.groupby(["product", "condition", "rank"], as_index=False).agg(
            observed_median_price=("realized_price", "median"),
            observed_sales=("realized_price", "size"),
        ),
        on=["product", "condition", "rank"],
        how="left",
    )

    result["observed_sales"] = result["observed_sales"].fillna(0).astype(int)
    result["has_observed_sale_at_condition"] = result["observed_sales"] > 0

    result["evidence_level"] = np.select(
        [
            result["observed_sales"] >= 3,
            result["observed_sales"].between(1, 2),
            (result["observed_sales"] == 0) & (result["slope_source"] == "product_specific"),
            (result["observed_sales"] == 0) & (result["slope_source"] == "global"),
        ],
        [
            "observed_stronger",
            "observed_thin",
            "estimated_from_product_curve",
            "estimated_from_global_curve",
        ],
        default="estimated"
    )

    return result.sort_values(["product", "rank"]).reset_index(drop=True)


price_ladder = build_condition_price_ladder(
    sales=sales,
    ladder=ladder,
    global_slope=slope_info["global_slope"],
)

price_ladder.head(20)

,product,condition,rank,estimated_price,slope_used,slope_source,anchor_rank,product_observed_sales,observed_median_price,observed_sales,has_observed_sale_at_condition,evidence_level
0,Collectible Card Z,Poor,1.0,100.1,0.099947,product_specific,3.0,4,116.71,1,True,observed_thin
1,Collectible Card Z,Fair,2.0,110.6,0.099947,product_specific,3.0,4,117.45,1,True,observed_thin
2,Collectible Card Z,Good,3.0,122.2,0.099947,product_specific,3.0,4,127.16,1,True,observed_thin
3,Collectible Card Z,Very Good,4.0,135.1,0.099947,product_specific,3.0,4,NaN,0,False,estimated_from_product_curve
4,Collectible Card Z,Excellent,5.0,149.2,0.099947,product_specific,3.0,4,NaN,0,False,estimated_from_product_curve
5,Collectible Card Z,Near Mint,6.0,164.9,0.099947,product_specific,3.0,4,186.46,1,True,observed_thin
6,Signed Print Q,Poor,1.0,432.2,0.071484,product_specific,3.5,4,443.14,1,True,observed_thin
7,Signed Print Q,Fair,2.0,464.3,0.071484,product_specific,3.5,4,479.69,1,True,observed_thin
8,Signed Print Q,Good,3.0,498.7,0.071484,product_specific,3.5,4,NaN,0,False,estimated_from_product_curve
9,Signed Print Q,Very Good,4.0,535.6,0.071484,product_specific,3.5,4,NaN,0,False,estimated_from_product_curve


## Read the evidence level

The output includes an `evidence_level` column.

- `observed_stronger`: at least 3 realized sales at that product-condition point
- `observed_thin`: 1 to 2 realized sales at that point
- `estimated_from_product_curve`: missing condition estimated from that product's own observed condition spread
- `estimated_from_global_curve`: missing condition estimated from the pooled auction condition curve

The last one is not useless. It just deserves a normal amount of suspicion.

In [6]:
# ============================================================
# 6. SUMMARY TABLES
# ============================================================

condition_summary = (
    sales.groupby(["condition", "rank"], as_index=False)
    .agg(
        realized_sales=("realized_price", "size"),
        median_realized_price=("realized_price", "median"),
        mean_realized_price=("realized_price", "mean"),
        min_realized_price=("realized_price", "min"),
        max_realized_price=("realized_price", "max"),
    )
    .sort_values("rank")
)

product_summary = (
    sales.groupby("product", as_index=False)
    .agg(
        realized_sales=("realized_price", "size"),
        observed_conditions=("condition", "nunique"),
        median_realized_price=("realized_price", "median"),
        min_realized_price=("realized_price", "min"),
        max_realized_price=("realized_price", "max"),
    )
    .sort_values(["observed_conditions", "realized_sales"], ascending=[True, True])
)

model_summary = pd.DataFrame([{
    **slope_info,
    "condition_step_percent_change": round((math.exp(slope_info["global_slope"]) - 1) * 100, 2),
    "products": sales["product"].nunique(),
    "realized_sales": len(sales),
    "conditions_in_ladder": len(ladder),
}])

display(model_summary)
display(condition_summary)
display(product_summary)

,global_slope,slope_source,n_pairwise_slopes,condition_step_percent_change,products,realized_sales,conditions_in_ladder
0,0.07933,observed_pairwise_median,20,8.26,6,18,6


,condition,rank,realized_sales,median_realized_price,mean_realized_price,min_realized_price,max_realized_price
4,Poor,1,2,279.925,279.925000,116.71,443.14
1,Fair,2,3,479.690,471.006667,117.45,815.88
2,Good,3,1,127.160,127.160000,127.16,127.16
5,Very Good,4,4,623.475,743.145000,248.06,1477.57
0,Excellent,5,4,790.895,855.282500,468.65,1370.69
3,Near Mint,6,4,367.145,394.900000,186.46,658.85


,product,realized_sales,observed_conditions,median_realized_price,min_realized_price,max_realized_price
2,Vintage Camera A,2,2,237.370,226.68,248.06
5,Watch Model Y,2,2,1424.130,1370.69,1477.57
3,Vintage Camera B,3,3,468.650,391.60,507.61
4,Watch Model X,3,3,855.350,815.88,1024.95
0,Collectible Card Z,4,4,122.305,116.71,186.46
1,Signed Print Q,4,4,518.265,443.14,658.85


In [7]:
# ============================================================
# 7. OPTIONAL: PRICE A NEW AUCTION ITEM
# ============================================================

def estimate_item_price(product: str, condition: str, price_ladder: pd.DataFrame) -> pd.DataFrame:
    match = price_ladder[
        (price_ladder["product"].astype(str) == str(product))
        & (price_ladder["condition"].astype(str) == str(condition))
    ].copy()

    if match.empty:
        raise ValueError(
            f"No estimate found for product={product!r}, condition={condition!r}. "
            "Check spelling against product names and condition ladder. The model is not fluent in vibes."
        )

    cols = [
        "product",
        "condition",
        "rank",
        "estimated_price",
        "observed_median_price",
        "observed_sales",
        "evidence_level",
        "slope_source",
    ]
    return match[cols]


# Example:
estimate_item_price(
    product=price_ladder["product"].iloc[0],
    condition=ladder.sort_values("rank")["condition"].iloc[-1],
    price_ladder=price_ladder,
)

,product,condition,rank,estimated_price,observed_median_price,observed_sales,evidence_level,slope_source
5,Collectible Card Z,Near Mint,6.0,164.9,186.46,1,observed_thin,product_specific


In [8]:
# ============================================================
# 8. EXPORT TO EXCEL
# ============================================================

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    price_ladder.to_excel(writer, sheet_name="Condition Price Ladder", index=False)
    model_summary.to_excel(writer, sheet_name="Model Summary", iwantndex=False)
    condition_summary.to_excel(writer, sheet_name="Condition Summary", index=False)
    product_summary.to_excel(writer, sheet_name="Product Summary", index=False)
    sales.to_excel(writer, sheet_name="Input Sales", index=False)

print(f"Exported: {OUTPUT_EXCEL}")

Exported: auction_condition_ladder_pricing_output.xlsx


## Limitations

This notebook is useful, but it is not an oracle.

It does not automatically account for:

- Auction venue differences
- Seller reputation
- Lot photography quality
- Bundle effects
- Buyer urgency
- Hidden damage
- Edition, size, year, or variant differences if your product grouping ignores them
- The auction gremlin that appears whenever two bidders decide they both deserve the same object

Use the output as a pricing ladder, reserve-price starting point, bid-support table, or listing guide.

Do not use it as a substitute for judgment. Judgment is still employed here, unfortunately.

## TaskMarket angle

This notebook is an example of the type of reusable business workflow TaskMarket is meant to support.

A useful analytical workflow should be packaged clearly enough that another analyst, operator, or agent can:

- Understand the assumptions
- Run the workflow
- Validate the output
- Modify the ladder
- Improve the method
- Reuse it without needing to inherit someone else's spreadsheet folklore

More practical workflows. Fewer haunted spreadsheets.

Built as a TaskMarket-style reusable work product.
